<a href="https://colab.research.google.com/github/servetgencimli-dotcom/DataScience/blob/main/Build%20and%20Query%20a%20Feature%20Store%20for%20ML%20Training%20%26%20Serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)
csv_files = os.listdir(path)
print("Files in the path:", csv_files)
df = pd.read_csv(os.path.join(path, csv_files[0]))

Using Colab cache for faster access to the 'paysim1' dataset.
Path to dataset files: /kaggle/input/paysim1
Files in the path: ['PS_20174392719_1491204439457_log.csv']


In [ ]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
df.shape

(6362620, 11)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [ ]:
df.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [ ]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
import os

df["customer_id"] = df["nameOrig"]

# timestamp
df["event_timestamp"] = pd.Timestamp.now()

# features
df["balance_delta"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

df["is_cashout"] = (
    df["type"] == "CASH_OUT"
).astype(int)

# aggregation
customer_features = (
    df.groupby("customer_id")
    .agg({
        "amount": "mean",
        "step": "count",
        "is_cashout": "sum",
        "balance_delta": "mean"
    })
    .reset_index()
)

customer_features.columns = [
    "customer_id",
    "avg_amount",
    "transaction_count",
    "cashout_count",
    "avg_balance_delta"
]

customer_features["event_timestamp"] = pd.Timestamp.now()

# Create the directory if it doesn't exist
os.makedirs("data", exist_ok=True)

customer_features.to_parquet(
    "data/customer_features.parquet",
    index=False)
customer_features.head()

,customer_id,avg_amount,transaction_count,cashout_count,avg_balance_delta,event_timestamp
0,C1000000639,244486.46,1,1,8946.00,2026-05-13 06:27:53.515515
1,C1000001337,3170.28,1,0,3170.28,2026-05-13 06:27:53.515515
2,C1000001725,8424.74,1,0,783.00,2026-05-13 06:27:53.515515
3,C1000002591,261877.19,1,0,-261877.19,2026-05-13 06:27:53.515515
4,C1000003372,20528.65,1,0,-20528.65,2026-05-13 06:27:53.515515


In [ ]:
!mkdir feature_repo

mkdir: cannot create directory ‘feature_repo’: File exists


In [ ]:
yaml_content = """
project: fraud_detection

registry: data/registry.db

provider: local

online_store:
    type: sqlite
    path: data/online_store.db
"""

with open(
    "/content/feature_repo/feature_store.yaml",
    "w"
) as f:
    f.write(yaml_content)

In [ ]:
entities_code = """
from feast import Entity

customer = Entity(
    name="customer_id",
    join_keys=["customer_id"],
)
"""

with open(
    "/content/feature_repo/entities.py",
    "w"
) as f:
    f.write(entities_code)

In [ ]:
source_code = """
from feast import FileSource

customer_source = FileSource(
    path="/content/data/customer_features.parquet",
    timestamp_field="event_timestamp",
)
"""

with open(
    "/content/feature_repo/data_sources.py",
    "w"
) as f:
    f.write(source_code)

In [ ]:
view_code = """
from datetime import timedelta

from feast import FeatureView, Field
from feast.types import Float32, Int64

from entities import customer
from data_sources import customer_source

customer_features_view = FeatureView(
    name="customer_features",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="avg_amount", dtype=Float32),
        Field(name="transaction_count", dtype=Int64),
        Field(name="cashout_count", dtype=Int64),
        Field(name="avg_balance_delta", dtype=Float32),
    ],
    online=True,
    source=customer_source,
)
"""

with open(
    "/content/feature_repo/feature_views.py",
    "w"
) as f:
    f.write(view_code)

In [ ]:
customer_features.head()

,customer_id,avg_amount,transaction_count,cashout_count,avg_balance_delta,event_timestamp
0,C1000000639,244486.46,1,1,8946.00,2026-05-13 06:27:53.515515
1,C1000001337,3170.28,1,0,3170.28,2026-05-13 06:27:53.515515
2,C1000001725,8424.74,1,0,783.00,2026-05-13 06:27:53.515515
3,C1000002591,261877.19,1,0,-261877.19,2026-05-13 06:27:53.515515
4,C1000003372,20528.65,1,0,-20528.65,2026-05-13 06:27:53.515515


In [ ]:
!feast feature-views list

Can't find feature repo configuration file at /content/feature_store.yaml. Make sure you're running feast from an initialized feast repository.


In [ ]:

customer_features['customer_id'].head()

,customer_id
0,C1000000639
1,C1000001337
2,C1000001725
3,C1000002591
4,C1000003372


In [ ]:
customer_features["event_timestamp"] = pd.Timestamp("2024-01-01")
customer_features.to_parquet("data/customer_features.parquet", index=False)

In [ ]:
# import os
# os.kill(os.getpid(), 9)

In [ ]:
# 1. Əvvəlcə bunu işlət - runtime-ı restart edəcək
!pip install -q feast==0.40.0 pyarrow==14.0.1 pandas scikit-learn mlflow
from feast import FeatureStore
store = FeatureStore(repo_path="/content/feature_repo")
print(store.list_feature_views())

/usr/local/lib/python3.12/dist-packages/feast/repo_config.py:229: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[<FeatureView(name = customer_features, entities = ['customer_id'], ttl = 30 days, 0:00:00, stream_source = None, batch_source = {
  "type": "BATCH_FILE",
  "timestampField": "event_timestamp",
  "fileOptions": {
    "uri": "/content/data/customer_features.parquet"
  },
  "name": "/content/data/customer_features.parquet"
}, entity_columns = [customer_id-String], features = [avg_amount-Float32, transaction_count-Int64, cashout_count-Int64, avg_balance_delta-Float32], description = , tags = {}, owner = , projection = FeatureViewProjection(name='customer_features', name_alias=None, desired_features=[], features=[avg_amount-Float32, transaction_count-Int64, cashout_count-Int64, avg_balance_delta-Float32], join_key_map={}), created_timestamp = 2026-05-13 05:30:03.752648, last_updated_timestamp = 2026-05-13 06:09:08.977061, online = True, materialization_intervals = [(datetime.datetime(2023, 1, 1, 0, 0, tzinfo=<UTC>), datetime.datetime(2026, 5, 9, 23, 59, 59, tzinfo=<UTC>)), (datetime.dateti

In [ ]:
df_check = pd.read_parquet("/content/data/customer_features.parquet")
df_small = df_check.head().copy()
df_small["event_timestamp"] = pd.Timestamp("2024-01-01", tz="UTC")
df_small.to_parquet("/content/data/customer_features.parquet", index=False)

print(f"Ölçü: {len(df_small)}")
print(df_small[df_small["customer_id"] == "C1000001337"])

Ölçü: 5
   customer_id  avg_amount  transaction_count  cashout_count  \
1  C1000001337     3170.28                  1              0   

   avg_balance_delta           event_timestamp  
1            3170.28 2024-01-01 00:00:00+00:00  


/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# 2. DB-ləri sil
import os
for f in ["/content/data/registry.db", "/content/data/online_store.db"]:
    if os.path.exists(f):
        os.remove(f)

In [ ]:
# 3. Apply + materialize (indi tez bitəcək)
%cd /content/feature_repo
!feast apply
!feast materialize 2023-01-01T00:00:00 2026-05-09T23:59:59

/content/feature_repo
/usr/local/lib/python3.12/dist-packages/feast/repo_config.py:229: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(
05/13/2026 06:28:17 AM root WARNING: Cannot use sqlite_vec for vector search
05/13/2026 06:28:17 AM root WARNING: Cannot use sqlite_vec for vector search
05/13/2026 06:28:17 AM root WARNING: Cannot use sqlite_vec for vector search
No changes to registry
No changes to infrastructure
/usr/local/lib/python3.12/dist-packages/feast/repo_config.py:229: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields 

In [ ]:
# 4. Sorğu
from feast import FeatureStore
store = FeatureStore(repo_path="/content/feature_repo")

features = store.get_online_features(
    features=["customer_features:avg_amount",
              "customer_features:transaction_count",
              "customer_features:cashout_count"
              ],
    entity_rows=[{"customer_id": "C1000001337"}]
).to_dict()
print(features)

/usr/local/lib/python3.12/dist-packages/feast/repo_config.py:229: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(


{'customer_id': ['C1000001337'], 'cashout_count': [0], 'avg_amount': [3170.280029296875], 'transaction_count': [1]}


/usr/local/lib/python3.12/dist-packages/feast/infra/online_stores/sqlite.py:220: DeprecationWarning: The default timestamp converter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  rows = cur.fetchall()


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score,confusion_matrix

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 1. Tranzaksiya-level features (hər tranzaksiya bir sətir)
fraud_data = df.copy()
# Yeni features
fraud_data["balance_diff_orig"] = fraud_data["oldbalanceOrg"] - fraud_data["newbalanceOrig"]
fraud_data["balance_diff_dest"] = fraud_data["newbalanceDest"] - fraud_data["oldbalanceDest"]
fraud_data["amount_ratio_orig"] = fraud_data["amount"] / (fraud_data["oldbalanceOrg"] + 1)
fraud_data["is_zero_balance_after"] = (fraud_data["newbalanceOrig"] == 0).astype(int)

features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "balance_diff_orig",
    "balance_diff_dest",
    "amount_ratio_orig",
    "is_zero_balance_after",
    "step"
]

X = fraud_data[features].fillna(0)
y = fraud_data["isFraud"]

print(y.value_counts())
fraud_only = fraud_data[fraud_data["isFraud"] == 1]

print(
    (fraud_only["balance_diff_orig"] == fraud_only["amount"]).mean()
)

isFraud
0    6354407
1       8213
Name: count, dtype: int64
0.991598685011567


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.utils import resample

# 1. Yalnız pre-transaction features (leakage yoxdur)


# Balanslaşdırma yoxdur - bütün data
train = fraud_data[fraud_data["step"] < 600]
test  = fraud_data[fraud_data["step"] >= 600]

print(f"Train: {len(train)}, Test: {len(test)}")
print(f"Train fraud: {train['isFraud'].sum()}")
print(f"Test fraud: {test['isFraud'].sum()}")

X_train = train[features].fillna(0)
y_train = train["isFraud"]
X_test  = test[features].fillna(0)
y_test  = test["isFraud"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

Train: 6259029, Test: 103591
Train fraud: 6595
Test fraud: 1618


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    101973
           1       1.00      1.00      1.00      1618

    accuracy                           1.00    103591
   macro avg       1.00      1.00      1.00    103591
weighted avg       1.00      1.00      1.00    103591

ROC-AUC: 1.0000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
!pip install mlflow -q

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, classification_report
import json

# 2. Experiment başlat
mlflow.set_experiment("fraud_detection")

with mlflow.start_run(run_name="random_forest_v1"):

    # Parametrləri qeyd et
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("sampling_ratio", "10:1")
    mlflow.log_param("features", features)

    # Metrikaları qeyd et
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba))
    mlflow.log_metric("accuracy", model.score(X_test, y_test))

    # Modeli qeyd et
    mlflow.sklearn.log_model(
        model,
        "fraud_model",
        registered_model_name="fraud_detector"
    )

    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")
    print("Model MLflow-a qeyd edildi!")

2026/05/13 06:46:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026/05/13 06:46:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: c4fadbfd619541dcb34c34130411b71a
Model MLflow-a qeyd edildi!


Registered model 'fraud_detector' already exists. Creating a new version of this model...
Created version '3' of model 'fraud_detector'.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import pickle
import os

# 3. Modeli fayla saxla (FastAPI üçün)
os.makedirs("/content/model", exist_ok=True)

with open("/content/model/fraud_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Feature siyahısını da saxla
with open("/content/model/features.json", "w") as f:
    json.dump(features, f)

print("Model saxlanıldı: /content/model/fraud_model.pkl")

Model saxlanıldı: /content/model/fraud_model.pkl


In [ ]:
# 1. FastAPI qur
!pip install fastapi uvicorn pyngrok -q

In [ ]:
# 2. API faylını yarat
api_code = '''
import pickle
import json
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel

# Model və features-i yüklə
with open("/content/model/fraud_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("/content/model/features.json", "r") as f:
    features = json.load(f)

app = FastAPI(title="Fraud Detection API")

class Transaction(BaseModel):
    amount: float
    type_encoded: int        # 0=CASH_IN, 1=CASH_OUT, 2=DEBIT, 3=PAYMENT, 4=TRANSFER
    oldbalanceOrg: float
    newbalanceOrig: float
    oldbalanceDest: float
    newbalanceDest: float
    step: int = 1

@app.get("/")
def root():
    return {"status": "Fraud Detection API işləyir"}

@app.post("/predict")
def predict(transaction: Transaction):
    # Features hesabla
    balance_diff_orig = transaction.oldbalanceOrg - transaction.newbalanceOrig
    balance_diff_dest = transaction.newbalanceDest - transaction.oldbalanceDest
    amount_ratio_orig = transaction.amount / (transaction.oldbalanceOrg + 1)
    is_zero_balance_after = int(transaction.newbalanceOrig == 0)

    X = [[
        transaction.amount,
        transaction.type_encoded,
        transaction.oldbalanceOrg,
        transaction.newbalanceOrig,
        transaction.oldbalanceDest,
        transaction.newbalanceDest,
        balance_diff_orig,
        balance_diff_dest,
        amount_ratio_orig,
        is_zero_balance_after,
        transaction.step
    ]]

    prediction = model.predict(X)[0]
    probability = model.predict_proba(X)[0][1]

    return {
        "is_fraud": bool(prediction),
        "fraud_probability": round(float(probability), 4),
        "risk_level": "HIGH" if probability > 0.7 else "MEDIUM" if probability > 0.3 else "LOW"
    }
'''

with open("/content/fraud_api.py", "w") as f:
    f.write(api_code)

print("API fayli yaradildi!")

API fayli yaradildi!


In [ ]:
# 3. API-yi background-da başlat
import subprocess
import time

process = subprocess.Popen(
    ["uvicorn", "fraud_api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content"
)
time.sleep(3)
print("API başladı: http://localhost:8000")

API başladı: http://localhost:8000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# 4. Test et
import requests

# Normal tranzaksiya
normal = {
    "amount": 100.0,
    "type_encoded": 3,
    "oldbalanceOrg": 5000.0,
    "newbalanceOrig": 4900.0,
    "oldbalanceDest": 1000.0,
    "newbalanceDest": 1100.0,
    "step": 1
}

# Şübhəli tranzaksiya (bütün balansı köçürür)
suspicious = {
    "amount": 50000.0,
    "type_encoded": 1,
    "oldbalanceOrg": 50000.0,
    "newbalanceOrig": 0.0,
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
    "step": 1
}

r1 = requests.post("http://localhost:8000/predict", json=normal)
r2 = requests.post("http://localhost:8000/predict", json=suspicious)

print("Normal tranzaksiya:", r1.json())
print("Şübhəli tranzaksiya:", r2.json())